In [3]:
import random
import shutil
from pathlib import Path

# =========================================================
# CONFIG
# =========================================================
SOURCE_ROOT = Path("/kaggle/input/datasets/gurjeetkaurmangat/kidney-ultrasound-images-stone-and-no-stone/my dataset final 512x512(implemented)")   # your dataset root base
DEST_ROOT = Path("/kaggle/working/kidney_ultrasound")

SAMPLES_PER_CLASS = 250
RANDOM_SEED = 42

VALID_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

random.seed(RANDOM_SEED)

# =========================================================
# HELPERS
# =========================================================
def find_class_folder(root: Path, class_name: str):
    matches = []
    for p in root.rglob("*"):
        if p.is_dir() and p.name.lower() == class_name.lower():
            matches.append(p)

    if not matches:
        raise FileNotFoundError(f"Could not find folder named '{class_name}' under {root}")

    print(f"Found '{class_name}' folder at: {matches[0]}")
    return matches[0]

def get_images(folder: Path):
    images = []
    for p in folder.rglob("*"):
        if p.is_file() and p.suffix.lower() in VALID_EXTENSIONS:
            images.append(p)
    return images

def sample_images(src_folder: Path, n_samples: int):
    images = get_images(src_folder)
    print(f"Total images found in '{src_folder.name}': {len(images)}")

    if len(images) < n_samples:
        raise ValueError(
            f"Not enough images in {src_folder}. Required: {n_samples}, Found: {len(images)}"
        )

    return random.sample(images, n_samples)

# =========================================================
# FIND SOURCE FOLDERS
# =========================================================
normal_src = find_class_folder(SOURCE_ROOT, "Normal")
stone_src = find_class_folder(SOURCE_ROOT, "stone")

# =========================================================
# PREPARE DESTINATION
# =========================================================
if DEST_ROOT.exists():
    shutil.rmtree(DEST_ROOT)
DEST_ROOT.mkdir(parents=True, exist_ok=True)

# =========================================================
# SAMPLE IMAGES
# =========================================================
normal_selected = sample_images(normal_src, SAMPLES_PER_CLASS)
stone_selected = sample_images(stone_src, SAMPLES_PER_CLASS)

all_selected = []

for i, img_path in enumerate(normal_selected, 1):
    all_selected.append(("normal", i, img_path))

for i, img_path in enumerate(stone_selected, 1):
    all_selected.append(("stone", i, img_path))

# Shuffle everything so both classes are randomly mixed in one folder
random.shuffle(all_selected)

# =========================================================
# COPY TO ONE SINGLE FOLDER
# =========================================================
for idx, (label, _, img_path) in enumerate(all_selected, 1):
    new_name = f"{idx:04d}_{label}{img_path.suffix.lower()}"
    dst_path = DEST_ROOT / new_name
    shutil.copy2(img_path, dst_path)

print("\nDone!")
print(f"Total copied images: {len(list(DEST_ROOT.glob('*')))}")
print(f"Saved in: {DEST_ROOT}")

Found 'Normal' folder at: /kaggle/input/datasets/gurjeetkaurmangat/kidney-ultrasound-images-stone-and-no-stone/my dataset final 512x512(implemented)/Normal
Found 'stone' folder at: /kaggle/input/datasets/gurjeetkaurmangat/kidney-ultrasound-images-stone-and-no-stone/my dataset final 512x512(implemented)/stone
Total images found in 'Normal': 4414
Total images found in 'stone': 5002

Done!
Total copied images: 500
Saved in: /kaggle/working/kidney_ultrasound


In [4]:
!pip uninstall -y torch torchvision torchaudio
!pip install -U --no-cache-dir torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126

Found existing installation: torch 2.10.0+cu128
Uninstalling torch-2.10.0+cu128:
  Successfully uninstalled torch-2.10.0+cu128
Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128
Found existing installation: torchaudio 2.10.0+cu128
Uninstalling torchaudio-2.10.0+cu128:
  Successfully uninstalled torchaudio-2.10.0+cu128
Looking in indexes: https://download.pytorch.org/whl/cu126
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.8/296.8 MB 237.2 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 MB 275.7 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.7/897.7 kB 261.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.2/200.2 MB 263.4 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 292.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 244.5 MB/s eta 0:00:00a 0:00:01
    

#### Loading Models

In [5]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image, UnidentifiedImageError
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

# =========================================================
# CONFIG
# =========================================================
IMAGE_DIR = Path("/kaggle/working/kidney_ultrasound")
OUTPUT_DIR = Path("/kaggle/working/kidney_embeddings")

BATCH_SIZE = 32
IMG_SIZE = 224
NUM_WORKERS = 2
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

VALID_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Using device:", DEVICE)
print("Image dir exists:", IMAGE_DIR.exists())

# =========================================================
# DATASET
# =========================================================
class KidneyImageDataset(Dataset):
    def __init__(self, image_dir, transform=None):
        self.image_dir = Path(image_dir)
        self.transform = transform

        self.image_paths = sorted([
            p for p in self.image_dir.iterdir()
            if p.is_file() and p.suffix.lower() in VALID_EXTENSIONS
        ])

        if len(self.image_paths) == 0:
            raise ValueError(f"No valid images found in {self.image_dir}")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]

        try:
            image = Image.open(img_path).convert("RGB")
        except UnidentifiedImageError:
            raise ValueError(f"Invalid image file: {img_path}")

        if self.transform:
            image = self.transform(image)

        filename = img_path.name

        # optional label inference from filename
        lower_name = filename.lower()
        if "normal" in lower_name:
            label = 0
        elif "stone" in lower_name:
            label = 1
        else:
            label = -1

        return image, filename, label

# =========================================================
# TRANSFORMS
# =========================================================
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

dataset = KidneyImageDataset(IMAGE_DIR, transform=transform)
loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

print("Total images:", len(dataset))

# =========================================================
# MODEL: EfficientNet-B0 without classifier head
# =========================================================
weights = models.EfficientNet_B0_Weights.DEFAULT
base_model = models.efficientnet_b0(weights=weights)

# Remove classifier head
feature_extractor = torch.nn.Sequential(
    base_model.features,
    base_model.avgpool
).to(DEVICE)

feature_extractor.eval()

# EfficientNet-B0 final feature size = 1280
EMBED_DIM = 1280

print("Feature extractor ready.")
print("Expected embedding dim:", EMBED_DIM)

# =========================================================
# EXTRACT EMBEDDINGS
# =========================================================
all_embeddings = []
all_filenames = []
all_labels = []

with torch.no_grad():
    for images, filenames, labels in tqdm(loader, desc="Extracting embeddings"):
        images = images.to(DEVICE, non_blocking=True)

        features = feature_extractor(images)              # [B, 1280, 1, 1]
        features = torch.flatten(features, 1)             # [B, 1280]

        # Normalize embeddings (L2 normalization)
        features = F.normalize(features, p=2, dim=1)

        all_embeddings.append(features.cpu().numpy())
        all_filenames.extend(filenames)
        all_labels.extend(labels.numpy().tolist())

embeddings = np.concatenate(all_embeddings, axis=0)

print("Embeddings shape:", embeddings.shape)
print("Total filenames:", len(all_filenames))
print("Total labels:", len(all_labels))

# =========================================================
# SAVE OUTPUTS
# =========================================================
embeddings_path = OUTPUT_DIR / "embeddings.npy"
filenames_path = OUTPUT_DIR / "filenames.npy"
labels_path = OUTPUT_DIR / "labels.npy"
csv_path = OUTPUT_DIR / "embeddings_with_names.csv"

np.save(embeddings_path, embeddings)
np.save(filenames_path, np.array(all_filenames))
np.save(labels_path, np.array(all_labels))

# CSV for inspection
df = pd.DataFrame(embeddings)
df.insert(0, "filename", all_filenames)
df.insert(1, "label", all_labels)
df.to_csv(csv_path, index=False)

print("\nSaved files:")
print("Embeddings:", embeddings_path)
print("Filenames :", filenames_path)
print("Labels    :", labels_path)
print("CSV       :", csv_path)

Using device: cuda
Image dir exists: True
Total images: 500
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 122MB/s] 


Feature extractor ready.
Expected embedding dim: 1280


Extracting embeddings:   0%|          | 0/16 [00:00<?, ?it/s]

Embeddings shape: (500, 1280)
Total filenames: 500
Total labels: 500

Saved files:
Embeddings: /kaggle/working/kidney_embeddings/embeddings.npy
Filenames : /kaggle/working/kidney_embeddings/filenames.npy
Labels    : /kaggle/working/kidney_embeddings/labels.npy
CSV       : /kaggle/working/kidney_embeddings/embeddings_with_names.csv


In [7]:
import os
import numpy as np
from pathlib import Path
from PIL import Image, UnidentifiedImageError

import torch
import torch.nn.functional as F
from torchvision import transforms, models

# =========================================================
# CONFIG
# =========================================================
EMBEDDINGS_DIR = Path("/kaggle/working/kidney_embeddings")
QUERY_IMAGE_PATH = "/kaggle/input/datasets/ahmedshayan1/random-test/house.jpg"   # change this

IMG_SIZE = 224
TOP_K = 5
THRESHOLD = 0.50   # tune this experimentally
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =========================================================
# LOAD STORED EMBEDDINGS
# =========================================================
embeddings_path = EMBEDDINGS_DIR / "embeddings.npy"
filenames_path = EMBEDDINGS_DIR / "filenames.npy"
labels_path = EMBEDDINGS_DIR / "labels.npy"

stored_embeddings = np.load(embeddings_path)   # shape: [N, 1280]
stored_filenames = np.load(filenames_path, allow_pickle=True)
stored_labels = np.load(labels_path, allow_pickle=True)

print("Stored embeddings shape:", stored_embeddings.shape)
print("Total stored images:", len(stored_filenames))

# convert to torch tensor
stored_embeddings_tensor = torch.tensor(
    stored_embeddings, dtype=torch.float32, device=DEVICE
)

# =========================================================
# IMAGE TRANSFORM
# =========================================================
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# =========================================================
# BUILD FEATURE EXTRACTOR
# =========================================================
def build_feature_extractor():
    weights = models.EfficientNet_B0_Weights.DEFAULT
    base_model = models.efficientnet_b0(weights=weights)

    model = torch.nn.Sequential(
        base_model.features,
        base_model.avgpool
    )
    model = model.to(DEVICE)
    model.eval()
    return model

feature_extractor = build_feature_extractor()

# =========================================================
# QUERY EMBEDDING EXTRACTION
# =========================================================
def extract_query_embedding(image_path: str):
    image_path = Path(image_path)

    if not image_path.exists():
        raise FileNotFoundError(f"Query image not found: {image_path}")

    try:
        image = Image.open(image_path).convert("RGB")
    except UnidentifiedImageError:
        raise ValueError(f"Invalid image: {image_path}")

    image_tensor = transform(image).unsqueeze(0).to(DEVICE)  # [1, 3, H, W]

    with torch.no_grad():
        feat = feature_extractor(image_tensor)     # [1, 1280, 1, 1]
        feat = torch.flatten(feat, 1)              # [1, 1280]
        feat = F.normalize(feat, p=2, dim=1)       # L2 normalize

    return feat   # torch tensor [1, 1280]

# =========================================================
# INFERENCE FUNCTION
# =========================================================
def kidney_similarity_inference(
    query_image_path: str,
    stored_embeddings_tensor: torch.Tensor,
    stored_filenames,
    stored_labels,
    top_k: int = 5,
    threshold: float = 0.50
):
    # 1. Extract query embedding
    query_embedding = extract_query_embedding(query_image_path)   # [1, 1280]

    # 2. Cosine similarity with all stored kidney embeddings
    # Since both are normalized, dot product = cosine similarity
    similarities = torch.matmul(stored_embeddings_tensor, query_embedding.T).squeeze(1)  # [N]

    # 3. Top-k nearest similarities
    top_k = min(top_k, similarities.shape[0])
    top_scores, top_indices = torch.topk(similarities, k=top_k, largest=True)

    top_scores_np = top_scores.detach().cpu().numpy()
    top_indices_np = top_indices.detach().cpu().numpy()

    # 4. Average top-k similarity
    avg_topk_similarity = float(np.mean(top_scores_np))

    # 5. Threshold decision
    is_kidney = avg_topk_similarity >= threshold

    # 6. Build top matches
    top_matches = []
    for rank, (score, idx) in enumerate(zip(top_scores_np, top_indices_np), start=1):
        label_value = stored_labels[idx]

        if label_value == 0:
            label_name = "normal"
        elif label_value == 1:
            label_name = "stone"
        else:
            label_name = "unknown"

        top_matches.append({
            "rank": rank,
            "filename": str(stored_filenames[idx]),
            "label": label_name,
            "similarity": float(score)
        })

    result = {
        "query_image": str(query_image_path),
        "avg_top{}_similarity".format(top_k): avg_topk_similarity,
        "threshold": threshold,
        "is_kidney": bool(is_kidney),
        "top_matches": top_matches
    }

    return result

# =========================================================
# RUN INFERENCE
# =========================================================
result = kidney_similarity_inference(
    query_image_path=QUERY_IMAGE_PATH,
    stored_embeddings_tensor=stored_embeddings_tensor,
    stored_filenames=stored_filenames,
    stored_labels=stored_labels,
    top_k=TOP_K,
    threshold=THRESHOLD
)

print("\n=== INFERENCE RESULT ===")
print("Query image:", result["query_image"])
print(f"Average Top-{TOP_K} Similarity:", result[f"avg_top{TOP_K}_similarity"])
print("Threshold:", result["threshold"])
print("Kidney decision:", result["is_kidney"])

print("\nTop matches:")
for item in result["top_matches"]:
    print(item)

Stored embeddings shape: (500, 1280)
Total stored images: 500

=== INFERENCE RESULT ===
Query image: /kaggle/input/datasets/ahmedshayan1/random-test/house.jpg
Average Top-5 Similarity: 0.16639827191829681
Threshold: 0.5
Kidney decision: False

Top matches:
{'rank': 1, 'filename': '0182_stone.jpg', 'label': 'stone', 'similarity': 0.16873443126678467}
{'rank': 2, 'filename': '0470_stone.jpg', 'label': 'stone', 'similarity': 0.16672660410404205}
{'rank': 3, 'filename': '0303_stone.jpg', 'label': 'stone', 'similarity': 0.16591580212116241}
{'rank': 4, 'filename': '0006_stone.jpg', 'label': 'stone', 'similarity': 0.1656094491481781}
{'rank': 5, 'filename': '0068_stone.jpg', 'label': 'stone', 'similarity': 0.1650051474571228}


#### Healthy and Unhealthy Detection

In [8]:
print("hllow")

hllow


#### Data set creation

In [9]:
import os
import shutil
from pathlib import Path

# =========================================================
# SOURCE PATHS
# =========================================================
NORMAL_SRC = Path(r"/kaggle/input/datasets/makiw4/kidney-dataset-stone-and-normal/Kidney Ultrasound Images Stone and No Stone/kidney US images/my dataset final 512x512(implemented)/Normal")

DEFECTED_SRC_1 = Path(r"/kaggle/input/datasets/makiw4/kidney-dataset-stone-and-normal/Kidney Ultrasound Images Stone and No Stone/kidney US images/my dataset final 512x512(implemented)/stone")
DEFECTED_SRC_2 = Path(r"/kaggle/input/datasets/ahmedshayan1/normal-kidney-dataset/kidneyUS_images_25_june_2025")

# =========================================================
# DESTINATION PATHS
# =========================================================
DEST_ROOT = Path("/kaggle/working/Kidney")
NORMAL_DEST = DEST_ROOT / "normal_Kidney"
DEFECTED_DEST = DEST_ROOT / "defected_Kidney"

# =========================================================
# CREATE OUTPUT FOLDERS
# =========================================================
NORMAL_DEST.mkdir(parents=True, exist_ok=True)
DEFECTED_DEST.mkdir(parents=True, exist_ok=True)

# =========================================================
# VALID IMAGE EXTENSIONS
# =========================================================
VALID_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

# =========================================================
# COPY FUNCTION
# =========================================================
def copy_images(src_folder: Path, dst_folder: Path, prefix: str):
    if not src_folder.exists():
        print(f"Source folder not found: {src_folder}")
        return 0

    count = 0
    for img_path in src_folder.rglob("*"):
        if img_path.is_file() and img_path.suffix.lower() in VALID_EXTS:
            new_name = f"{prefix}_{count:06d}{img_path.suffix.lower()}"
            dst_path = dst_folder / new_name
            shutil.copy2(img_path, dst_path)
            count += 1

    print(f"Copied {count} images from: {src_folder}")
    return count

# =========================================================
# MERGE DATASETS
# =========================================================
normal_count = copy_images(NORMAL_SRC, NORMAL_DEST, "normal")

defected_count_1 = copy_images(DEFECTED_SRC_1, DEFECTED_DEST, "stone")

# continue numbering for second defected source
start_idx = defected_count_1

if DEFECTED_SRC_2.exists():
    count = 0
    for img_path in DEFECTED_SRC_2.rglob("*"):
        if img_path.is_file() and img_path.suffix.lower() in VALID_EXTS:
            new_name = f"defected_{start_idx + count:06d}{img_path.suffix.lower()}"
            dst_path = DEFECTED_DEST / new_name
            shutil.copy2(img_path, dst_path)
            count += 1
    print(f"Copied {count} images from: {DEFECTED_SRC_2}")
    defected_count_2 = count
else:
    print(f"Source folder not found: {DEFECTED_SRC_2}")
    defected_count_2 = 0

# =========================================================
# FINAL SUMMARY
# =========================================================
print("\nDone")
print(f"Total Normal images stored in   : {NORMAL_DEST} = {normal_count}")
print(f"Total Defected images stored in : {DEFECTED_DEST} = {defected_count_1 + defected_count_2}")

Copied 4414 images from: /kaggle/input/datasets/makiw4/kidney-dataset-stone-and-normal/Kidney Ultrasound Images Stone and No Stone/kidney US images/my dataset final 512x512(implemented)/Normal
Copied 5002 images from: /kaggle/input/datasets/makiw4/kidney-dataset-stone-and-normal/Kidney Ultrasound Images Stone and No Stone/kidney US images/my dataset final 512x512(implemented)/stone
Copied 534 images from: /kaggle/input/datasets/ahmedshayan1/normal-kidney-dataset/kidneyUS_images_25_june_2025

Done
Total Normal images stored in   : /kaggle/working/Kidney/normal_Kidney = 4414
Total Defected images stored in : /kaggle/working/Kidney/defected_Kidney = 5536


In [10]:
# =========================================================
# Kidney Ultrasound Binary Classification Training Pipeline
# EfficientNet-B3 + tqdm
# Dataset:
# /kaggle/working/Kidney/normal_Kidney
# /kaggle/working/Kidney/defected_Kidney
# =========================================================

import os
import copy
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, UnidentifiedImageError
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models

warnings.filterwarnings("ignore")

# =========================================================
# CONFIG
# =========================================================
NORMAL_DIR = "/kaggle/working/Kidney/normal_Kidney"
DEFECTED_DIR = "/kaggle/working/Kidney/defected_Kidney"

OUTPUT_DIR = "/kaggle/working/kidney_effb3_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

SEED = 42
IMG_SIZE = 300
BATCH_SIZE = 16
NUM_WORKERS = 1
HEAD_EPOCHS = 5
FINETUNE_EPOCHS = 20
HEAD_LR = 1e-3
FINETUNE_LR = 1e-4
WEIGHT_DECAY = 1e-4
PATIENCE = 5

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

CLASS_TO_IDX = {
    "normal_Kidney": 0,
    "defected_Kidney": 1
}
IDX_TO_CLASS = {v: k for k, v in CLASS_TO_IDX.items()}

# =========================================================
# SEED
# =========================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

# =========================================================
# COLLECT FILES
# =========================================================
VALID_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

def is_valid_image_file(path):
    return Path(path).suffix.lower() in VALID_EXTENSIONS

def collect_image_paths():
    data = []

    for img_name in os.listdir(NORMAL_DIR):
        img_path = os.path.join(NORMAL_DIR, img_name)
        if os.path.isfile(img_path) and is_valid_image_file(img_path):
            data.append((img_path, CLASS_TO_IDX["normal_Kidney"]))

    for img_name in os.listdir(DEFECTED_DIR):
        img_path = os.path.join(DEFECTED_DIR, img_name)
        if os.path.isfile(img_path) and is_valid_image_file(img_path):
            data.append((img_path, CLASS_TO_IDX["defected_Kidney"]))

    return data

all_data = collect_image_paths()
print("Total images found:", len(all_data))

if len(all_data) == 0:
    raise ValueError("No images found in dataset folders.")

df = pd.DataFrame(all_data, columns=["image_path", "label"])
print(df["label"].value_counts())

# =========================================================
# TRAIN / VAL / TEST SPLIT
# =========================================================
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df["label"],
    random_state=SEED
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=SEED
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

# =========================================================
# TRANSFORMS
# =========================================================
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.10, contrast=0.10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# =========================================================
# DATASET
# =========================================================
class KidneyDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        image_path = row["image_path"]
        label = int(row["label"])

        try:
            image = Image.open(image_path).convert("RGB")
        except (UnidentifiedImageError, OSError):
            image = Image.new("RGB", (IMG_SIZE, IMG_SIZE), (0, 0, 0))

        if self.transform:
            image = self.transform(image)

        return image, label

train_dataset = KidneyDataset(train_df, transform=train_transform)
val_dataset = KidneyDataset(val_df, transform=eval_transform)
test_dataset = KidneyDataset(test_df, transform=eval_transform)

# =========================================================
# WEIGHTED SAMPLER + CLASS WEIGHTS
# =========================================================
train_labels = train_df["label"].values
class_counts = np.bincount(train_labels)
class_weights = 1.0 / class_counts
sample_weights = [class_weights[label] for label in train_labels]

sampler = WeightedRandomSampler(
    weights=torch.DoubleTensor(sample_weights),
    num_samples=len(sample_weights),
    replacement=True
)

criterion_weights = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)

# =========================================================
# DATALOADERS
# =========================================================
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=sampler,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

# =========================================================
# MODEL
# =========================================================
def build_model(num_classes=2):
    model = models.efficientnet_b3(weights=models.EfficientNet_B3_Weights.IMAGENET1K_V1)
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(in_features, num_classes)
    )
    return model

model = build_model(num_classes=2).to(DEVICE)

# =========================================================
# FREEZE / UNFREEZE
# =========================================================
def freeze_backbone(model):
    for param in model.features.parameters():
        param.requires_grad = False
    for param in model.classifier.parameters():
        param.requires_grad = True

def unfreeze_all(model):
    for param in model.parameters():
        param.requires_grad = True

# =========================================================
# LOSS / OPTIMIZER / SCHEDULER
# =========================================================
criterion = nn.CrossEntropyLoss(weight=criterion_weights)

def get_optimizer(model, lr):
    return torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr,
        weight_decay=WEIGHT_DECAY
    )

# =========================================================
# METRICS
# =========================================================
def compute_metrics(y_true, y_pred, y_prob):
    metrics = {}
    metrics["accuracy"] = accuracy_score(y_true, y_pred)
    metrics["precision"] = precision_score(y_true, y_pred, zero_division=0)
    metrics["recall"] = recall_score(y_true, y_pred, zero_division=0)
    metrics["f1"] = f1_score(y_true, y_pred, zero_division=0)

    try:
        metrics["auc"] = roc_auc_score(y_true, y_prob)
    except:
        metrics["auc"] = 0.0

    return metrics

# =========================================================
# TRAIN / VALIDATE ONE EPOCH
# =========================================================
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()

    running_loss = 0.0
    all_labels = []
    all_preds = []
    all_probs = []

    progress_bar = tqdm(loader, desc="Training", leave=False)

    for images, labels in progress_bar:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        probs = torch.softmax(outputs, dim=1)[:, 1]
        preds = torch.argmax(outputs, dim=1)

        running_loss += loss.item() * images.size(0)
        all_labels.extend(labels.detach().cpu().numpy())
        all_preds.extend(preds.detach().cpu().numpy())
        all_probs.extend(probs.detach().cpu().numpy())

        progress_bar.set_postfix(loss=f"{loss.item():.4f}")

    epoch_loss = running_loss / len(loader.dataset)
    metrics = compute_metrics(all_labels, all_preds, all_probs)

    return epoch_loss, metrics

@torch.no_grad()
def validate_one_epoch(model, loader, criterion, device):
    model.eval()

    running_loss = 0.0
    all_labels = []
    all_preds = []
    all_probs = []

    progress_bar = tqdm(loader, desc="Validating", leave=False)

    for images, labels in progress_bar:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        outputs = model(images)
        loss = criterion(outputs, labels)

        probs = torch.softmax(outputs, dim=1)[:, 1]
        preds = torch.argmax(outputs, dim=1)

        running_loss += loss.item() * images.size(0)
        all_labels.extend(labels.detach().cpu().numpy())
        all_preds.extend(preds.detach().cpu().numpy())
        all_probs.extend(probs.detach().cpu().numpy())

        progress_bar.set_postfix(loss=f"{loss.item():.4f}")

    epoch_loss = running_loss / len(loader.dataset)
    metrics = compute_metrics(all_labels, all_preds, all_probs)

    return epoch_loss, metrics

# =========================================================
# TRAINING LOOP
# =========================================================
def run_training_phase(model, train_loader, val_loader, criterion, optimizer, scheduler,
                       device, epochs, phase_name, best_model_path, patience=5):
    best_f1 = -1.0
    best_state = copy.deepcopy(model.state_dict())
    patience_counter = 0
    history = []

    print(f"\n{'=' * 60}")
    print(f"Starting phase: {phase_name}")
    print(f"{'=' * 60}")

    for epoch in range(1, epochs + 1):
        print(f"\n[{phase_name}] Epoch {epoch}/{epochs}")

        train_loss, train_metrics = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_metrics = validate_one_epoch(model, val_loader, criterion, device)

        scheduler.step(val_loss)

        history_row = {
            "phase": phase_name,
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "train_acc": train_metrics["accuracy"],
            "val_acc": val_metrics["accuracy"],
            "train_precision": train_metrics["precision"],
            "val_precision": val_metrics["precision"],
            "train_recall": train_metrics["recall"],
            "val_recall": val_metrics["recall"],
            "train_f1": train_metrics["f1"],
            "val_f1": val_metrics["f1"],
            "train_auc": train_metrics["auc"],
            "val_auc": val_metrics["auc"],
        }
        history.append(history_row)

        print(
            f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}\n"
            f"Train Acc: {train_metrics['accuracy']:.4f} | Val Acc: {val_metrics['accuracy']:.4f}\n"
            f"Train Precision: {train_metrics['precision']:.4f} | Val Precision: {val_metrics['precision']:.4f}\n"
            f"Train Recall: {train_metrics['recall']:.4f} | Val Recall: {val_metrics['recall']:.4f}\n"
            f"Train F1: {train_metrics['f1']:.4f} | Val F1: {val_metrics['f1']:.4f}\n"
            f"Train AUC: {train_metrics['auc']:.4f} | Val AUC: {val_metrics['auc']:.4f}"
        )

        if val_metrics["f1"] > best_f1:
            best_f1 = val_metrics["f1"]
            best_state = copy.deepcopy(model.state_dict())
            torch.save(best_state, best_model_path)
            print(f"Best model saved to: {best_model_path}")
            patience_counter = 0
        else:
            patience_counter += 1
            print(f"No improvement. Patience: {patience_counter}/{patience}")

        if patience_counter >= patience:
            print("Early stopping triggered.")
            break

    model.load_state_dict(best_state)
    return model, history

# =========================================================
# PHASE 1: TRAIN HEAD ONLY
# =========================================================
freeze_backbone(model)

optimizer = get_optimizer(model, HEAD_LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=2
)

head_best_model_path = os.path.join(OUTPUT_DIR, "best_head_only_effb3.pth")

model, history_head = run_training_phase(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    device=DEVICE,
    epochs=HEAD_EPOCHS,
    phase_name="Head Training",
    best_model_path=head_best_model_path,
    patience=PATIENCE
)

# =========================================================
# PHASE 2: FINE-TUNE FULL MODEL
# =========================================================
unfreeze_all(model)

optimizer = get_optimizer(model, FINETUNE_LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=2
)

finetune_best_model_path = os.path.join(OUTPUT_DIR, "best_full_effb3.pth")

model, history_finetune = run_training_phase(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    device=DEVICE,
    epochs=FINETUNE_EPOCHS,
    phase_name="Full Fine-Tuning",
    best_model_path=finetune_best_model_path,
    patience=PATIENCE
)

# =========================================================
# SAVE HISTORY
# =========================================================
history_df = pd.DataFrame(history_head + history_finetune)
history_csv_path = os.path.join(OUTPUT_DIR, "training_history.csv")
history_df.to_csv(history_csv_path, index=False)
print(f"\nTraining history saved to: {history_csv_path}")

# =========================================================
# LOAD BEST MODEL FOR TESTING
# =========================================================
best_model = build_model(num_classes=2).to(DEVICE)
best_model.load_state_dict(torch.load(finetune_best_model_path, map_location=DEVICE))
best_model.eval()

# =========================================================
# EVALUATE ON TEST SET
# =========================================================
@torch.no_grad()
def evaluate_model(model, loader, device):
    model.eval()

    all_labels = []
    all_preds = []
    all_probs = []

    progress_bar = tqdm(loader, desc="Testing", leave=False)

    for images, labels in progress_bar:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)[:, 1]
        preds = torch.argmax(outputs, dim=1)

        all_labels.extend(labels.detach().cpu().numpy())
        all_preds.extend(preds.detach().cpu().numpy())
        all_probs.extend(probs.detach().cpu().numpy())

    metrics = compute_metrics(all_labels, all_preds, all_probs)
    cm = confusion_matrix(all_labels, all_preds)

    return metrics, cm, all_labels, all_preds, all_probs

test_metrics, cm, y_true, y_pred, y_prob = evaluate_model(best_model, test_loader, DEVICE)

print("\n" + "=" * 60)
print("FINAL TEST RESULTS")
print("=" * 60)
print(f"Accuracy : {test_metrics['accuracy']:.4f}")
print(f"Precision: {test_metrics['precision']:.4f}")
print(f"Recall   : {test_metrics['recall']:.4f}")
print(f"F1 Score : {test_metrics['f1']:.4f}")
print(f"ROC AUC  : {test_metrics['auc']:.4f}")

print("\nConfusion Matrix:")
print(cm)

print("\nClassification Report:")
print(classification_report(
    y_true,
    y_pred,
    target_names=["normal_Kidney", "defected_Kidney"],
    digits=4
))

# =========================================================
# SAVE METRICS
# =========================================================
metrics_txt_path = os.path.join(OUTPUT_DIR, "test_metrics.txt")
with open(metrics_txt_path, "w") as f:
    f.write("FINAL TEST RESULTS\n")
    f.write("=" * 40 + "\n")
    f.write(f"Accuracy : {test_metrics['accuracy']:.4f}\n")
    f.write(f"Precision: {test_metrics['precision']:.4f}\n")
    f.write(f"Recall   : {test_metrics['recall']:.4f}\n")
    f.write(f"F1 Score : {test_metrics['f1']:.4f}\n")
    f.write(f"ROC AUC  : {test_metrics['auc']:.4f}\n\n")
    f.write("Confusion Matrix:\n")
    f.write(str(cm) + "\n\n")
    f.write("Classification Report:\n")
    f.write(classification_report(
        y_true,
        y_pred,
        target_names=["normal_Kidney", "defected_Kidney"],
        digits=4
    ))

print(f"\nTest metrics saved to: {metrics_txt_path}")
print(f"Best model saved at: {finetune_best_model_path}")

Using device: cuda
Total images found: 9950
label
1    5536
0    4414
Name: count, dtype: int64
Train: 6965 | Val: 1492 | Test: 1493
Downloading: "https://download.pytorch.org/models/efficientnet_b3_rwightman-b3899882.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b3_rwightman-b3899882.pth


100%|██████████| 47.2M/47.2M [00:00<00:00, 169MB/s] 



Starting phase: Head Training

[Head Training] Epoch 1/5


Training:   0%|          | 0/436 [00:00<?, ?it/s]

Validating:   0%|          | 0/94 [00:00<?, ?it/s]

Train Loss: 0.2226 | Val Loss: 0.0772
Train Acc: 0.9269 | Val Acc: 0.9879
Train Precision: 0.9523 | Val Precision: 0.9892
Train Recall: 0.8982 | Val Recall: 0.9892
Train F1: 0.9245 | Val F1: 0.9892
Train AUC: 0.9802 | Val AUC: 0.9990
Best model saved to: /kaggle/working/kidney_effb3_output/best_head_only_effb3.pth

[Head Training] Epoch 2/5


... (31 outputs left out)

Validating:   0%|          | 0/94 [00:00<?, ?it/s]

Train Loss: 0.0003 | Val Loss: 0.0000
Train Acc: 0.9999 | Val Acc: 1.0000
Train Precision: 0.9997 | Val Precision: 1.0000
Train Recall: 1.0000 | Val Recall: 1.0000
Train F1: 0.9999 | Val F1: 1.0000
Train AUC: 1.0000 | Val AUC: 1.0000
No improvement. Patience: 5/5
Early stopping triggered.

Training history saved to: /kaggle/working/kidney_effb3_output/training_history.csv


Testing:   0%|          | 0/94 [00:00<?, ?it/s]


FINAL TEST RESULTS
Accuracy : 1.0000
Precision: 1.0000
Recall   : 1.0000
F1 Score : 1.0000
ROC AUC  : 1.0000

Confusion Matrix:
[[662   0]
 [  0 831]]

Classification Report:
                 precision    recall  f1-score   support

  normal_Kidney     1.0000    1.0000    1.0000       662
defected_Kidney     1.0000    1.0000    1.0000       831

       accuracy                         1.0000      1493
      macro avg     1.0000    1.0000    1.0000      1493
   weighted avg     1.0000    1.0000    1.0000      1493


Test metrics saved to: /kaggle/working/kidney_effb3_output/test_metrics.txt
Best model saved at: /kaggle/working/kidney_effb3_output/best_full_effb3.pth


In [12]:
print("data")

data


In [13]:
# =========================================================
# Kidney Ultrasound Binary Classification Inference Script
# For test dataset:
# /kaggle/input/datasets/ahmedshayan1/testing-dataset/Diseased
# /kaggle/input/datasets/ahmedshayan1/testing-dataset/Normal
# =========================================================

import os
import warnings
from pathlib import Path

import pandas as pd
import numpy as np
from PIL import Image, UnidentifiedImageError
from tqdm.auto import tqdm
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

warnings.filterwarnings("ignore")

# =========================================================
# CONFIG
# =========================================================
MODEL_PATH = "/kaggle/working/kidney_effb3_output/best_full_effb3.pth"

DISEASED_DIR = "/kaggle/input/datasets/ahmedshayan1/testing-dataset/Diseased"
NORMAL_DIR   = "/kaggle/input/datasets/ahmedshayan1/testing-dataset/Normal"

OUTPUT_DIR = "/kaggle/working/kidney_inference_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

IMG_SIZE = 300
BATCH_SIZE = 16
NUM_WORKERS = 2

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

CLASS_TO_IDX = {
    "Normal": 0,
    "Diseased": 1
}
IDX_TO_CLASS = {v: k for k, v in CLASS_TO_IDX.items()}

VALID_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

# =========================================================
# MODEL
# =========================================================
def build_model(num_classes=2):
    model = models.efficientnet_b3(weights=None)
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(in_features, num_classes)
    )
    return model

# =========================================================
# TRANSFORM
# =========================================================
test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# =========================================================
# DATA COLLECTION
# =========================================================
def is_valid_image_file(path):
    return Path(path).suffix.lower() in VALID_EXTENSIONS

def collect_test_data():
    data = []

    for img_name in os.listdir(NORMAL_DIR):
        img_path = os.path.join(NORMAL_DIR, img_name)
        if os.path.isfile(img_path) and is_valid_image_file(img_path):
            data.append((img_path, CLASS_TO_IDX["Normal"]))

    for img_name in os.listdir(DISEASED_DIR):
        img_path = os.path.join(DISEASED_DIR, img_name)
        if os.path.isfile(img_path) and is_valid_image_file(img_path):
            data.append((img_path, CLASS_TO_IDX["Diseased"]))

    return data

test_data = collect_test_data()
print("Total test images found:", len(test_data))

if len(test_data) == 0:
    raise ValueError("No test images found.")

test_df = pd.DataFrame(test_data, columns=["image_path", "label"])
print("\nClass distribution:")
print(test_df["label"].value_counts())

# =========================================================
# DATASET
# =========================================================
class KidneyTestDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        image_path = row["image_path"]
        label = int(row["label"])

        try:
            image = Image.open(image_path).convert("RGB")
        except (UnidentifiedImageError, OSError):
            image = Image.new("RGB", (IMG_SIZE, IMG_SIZE), (0, 0, 0))

        if self.transform:
            image = self.transform(image)

        return image, label, image_path

test_dataset = KidneyTestDataset(test_df, transform=test_transform)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

# =========================================================
# LOAD MODEL
# =========================================================
model = build_model(num_classes=2).to(DEVICE)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()

print(f"\nModel loaded from: {MODEL_PATH}")

# =========================================================
# INFERENCE
# =========================================================
@torch.no_grad()
def run_inference(model, loader, device):
    model.eval()

    all_labels = []
    all_preds = []
    all_probs = []
    all_paths = []

    progress_bar = tqdm(loader, desc="Running Inference")

    for images, labels, image_paths in progress_bar:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)
        preds = torch.argmax(outputs, dim=1)

        diseased_probs = probs[:, 1]

        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_probs.extend(diseased_probs.cpu().numpy())
        all_paths.extend(image_paths)

    return all_labels, all_preds, all_probs, all_paths

y_true, y_pred, y_prob, image_paths = run_inference(model, test_loader, DEVICE)

# =========================================================
# METRICS
# =========================================================
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, zero_division=0)
recall = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)

try:
    auc = roc_auc_score(y_true, y_prob)
except:
    auc = 0.0

cm = confusion_matrix(y_true, y_pred)
report = classification_report(
    y_true,
    y_pred,
    target_names=["Normal", "Diseased"],
    digits=4
)

# =========================================================
# RESULTS DATAFRAME
# =========================================================
results_df = pd.DataFrame({
    "image_path": image_paths,
    "true_label_idx": y_true,
    "pred_label_idx": y_pred,
    "true_label_name": [IDX_TO_CLASS[x] for x in y_true],
    "pred_label_name": [IDX_TO_CLASS[x] for x in y_pred],
    "diseased_probability": y_prob
})

results_csv_path = os.path.join(OUTPUT_DIR, "inference_results.csv")
results_df.to_csv(results_csv_path, index=False)

# =========================================================
# PRINT RESULTS
# =========================================================
print("\n" + "=" * 60)
print("FINAL INFERENCE RESULTS")
print("=" * 60)
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"ROC AUC  : {auc:.4f}")

print("\nConfusion Matrix:")
print(cm)

print("\nClassification Report:")
print(report)

# =========================================================
# SAVE METRICS
# =========================================================
metrics_txt_path = os.path.join(OUTPUT_DIR, "inference_metrics.txt")
with open(metrics_txt_path, "w") as f:
    f.write("FINAL INFERENCE RESULTS\n")
    f.write("=" * 50 + "\n")
    f.write(f"Accuracy : {accuracy:.4f}\n")
    f.write(f"Precision: {precision:.4f}\n")
    f.write(f"Recall   : {recall:.4f}\n")
    f.write(f"F1 Score : {f1:.4f}\n")
    f.write(f"ROC AUC  : {auc:.4f}\n\n")
    f.write("Confusion Matrix:\n")
    f.write(str(cm) + "\n\n")
    f.write("Classification Report:\n")
    f.write(report)

print(f"\nSaved predictions to: {results_csv_path}")
print(f"Saved metrics to    : {metrics_txt_path}")

Using device: cuda
Total test images found: 12

Class distribution:
label
1    7
0    5
Name: count, dtype: int64

Model loaded from: /kaggle/working/kidney_effb3_output/best_full_effb3.pth


Running Inference:   0%|          | 0/1 [00:00<?, ?it/s]


FINAL INFERENCE RESULTS
Accuracy : 0.5833
Precision: 0.6000
Recall   : 0.8571
F1 Score : 0.7059
ROC AUC  : 0.1714

Confusion Matrix:
[[1 4]
 [1 6]]

Classification Report:
              precision    recall  f1-score   support

      Normal     0.5000    0.2000    0.2857         5
    Diseased     0.6000    0.8571    0.7059         7

    accuracy                         0.5833        12
   macro avg     0.5500    0.5286    0.4958        12
weighted avg     0.5583    0.5833    0.5308        12


Saved predictions to: /kaggle/working/kidney_inference_results/inference_results.csv
Saved metrics to    : /kaggle/working/kidney_inference_results/inference_metrics.txt


In [14]:
import os
import shutil

folders_to_zip = [
    "/kaggle/working/kidney_effb3_output",
    "/kaggle/working/kidney_embeddings"
]

for folder_path in folders_to_zip:
    if os.path.exists(folder_path):
        zip_path = folder_path
        shutil.make_archive(zip_path, 'zip', folder_path)
        print(f"Created zip: {zip_path}.zip")
    else:
        print(f"Folder not found: {folder_path}")

Created zip: /kaggle/working/kidney_effb3_output.zip
Created zip: /kaggle/working/kidney_embeddings.zip


In [1]:
print("hellow")

hellow
